# Step 1 — ECG pipeline modernization: a supported narrative walkthrough — v1.2

This notebook is the repository's canonical explanation and orchestration layer. It presents a historical educational notebook project as a **modernization and reproducibility case study**: versioned provenance, explicit data contracts, subject-aware evaluation discipline, and auditable operational evidence. Implementation remains in `src/ecg_anomaly_detection/`, configuration, the `ecg-data` CLI, and tests.

> **Use boundary:** This repository is not medical software, clinical software, or production ML. It makes no clinical, diagnostic, deployment, or medical-utility claims. Validation evidence is not benchmark evidence.


> **STEP 0 PREREQUISITE**
>
> Complete [`00-environment-setup-and-artifact-generation.ipynb`](./00-environment-setup-and-artifact-generation.ipynb) before using generated local artifacts.
>
> This notebook explains the governed modernization lifecycle. It does **not** install the environment, clean local generated outputs, acquire data, or run the full pipeline. If Step 0 is blocked by PIPE-006, this notebook should still execute and report that generated-artifact evidence is incomplete rather than throwing a traceback.


## Version history and change log

| Version | Change |
|---:|---|
| 1.2 | Restored strict dependency behavior for the public workflow. Step 1 now requires Step 0 to complete artifact generation and verifies dataset, split, split-quality, and run-manifest artifacts before continuing. |
| 1.1 | Converts Step 0 readiness from a hard failure into a non-throwing status report so the narrative walkthrough can still execute when Step 0 ends in a controlled PIPE-006 blocked state. Adds visible internal comments to every Python cell, documents mutation boundaries, and keeps generated-artifact evidence inspection optional. |
| 1.0 | Initial public narrative walkthrough for the supported modernization lifecycle. |


> **CODE CELL FUNCTION**
>
> Run a compact Step 0 readiness check before continuing. The cell uses only Python standard-library modules and fails with a clear remediation message if the local checkout has not completed Step 0.

In [ ]:
# CODE CELL FUNCTION:
# Enforce Step 0 completion before the narrative walkthrough inspects generated state.
#
# COMMENT MAP:
# - Read the machine-readable Step 0 status written by notebook 00.
# - Require actual model-ready artifacts, not just a completed notebook run.
# - Fail fast when Step 0 stopped at PIPE-006 or any other pipeline error.
# - Keep this cell read-only so it verifies state without mutating local artifacts.
# - Preserve reviewer UX by making the missing dependency explicit at the top of Step 1.

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
STEP0_STATUS_PATH = ROOT / "notebooks" / "local" / "step0-pipeline-status.json"


def load_json_object(path: Path) -> dict[str, Any]:
    """Load a JSON object from disk and reject missing or malformed status artifacts."""
    if not path.is_file():
        raise RuntimeError(
            "Step 1 requires Step 0 to complete first, but the Step 0 status file is missing.\n"
            f"Expected: {path}\n"
            "Run notebooks/00-environment-setup-and-artifact-generation.ipynb from the top."
        )
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Step 0 status file is invalid JSON: {path}") from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f"Step 0 status file must contain a JSON object: {path}")
    return payload


def newest_dataset_index(root: Path) -> Path:
    """Return the newest generated dataset index produced by strict Step 0."""
    candidates = sorted(
        (root / "data" / "processed" / "runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise RuntimeError(
            "Step 0 has not produced data/processed/runs/<run-id>/dataset-index.json. "
            "Step 1 will not continue because the public workflow is not artifact-ready."
        )
    return candidates[-1]


def require_file(path: Path, label: str) -> Path:
    """Require one downstream artifact without attempting repair from Step 1."""
    if not path.is_file():
        raise RuntimeError(f"Missing required Step 0 artifact for Step 1: {label}: {path}")
    return path


status = load_json_object(STEP0_STATUS_PATH)
if status.get("status") != "complete":
    raise RuntimeError(
        "Step 1 requires Step 0 to complete artifact generation. Step 0 did not complete.\n"
        f"Status: {status.get('status')}\n"
        f"Reason: {status.get('reason')}\n"
        f"Message: {status.get('message')}\n"
        f"Log file: {status.get('log_file')}"
    )

index_path = newest_dataset_index(ROOT)
run_id = index_path.parent.name
run_root = ROOT / "artifacts" / "runs" / run_id
verified = {
    "dataset_index": require_file(index_path, "dataset index"),
    "split_manifest": require_file(run_root / "split.json", "split manifest"),
    "split_quality": require_file(run_root / "split_quality_summary.json", "split quality summary"),
    "run_manifest": require_file(run_root / "run-manifest.json", "run manifest"),
}

{
    "step_0_status": status.get("status"),
    "run_id": run_id,
    "verified_artifacts": {label: str(path.relative_to(ROOT)) for label, path in verified.items()},
    "narrative_walkthrough_ready": True,
    "test_partition_opened": False,
}


## 1. Repository purpose and modernization context

The supported workflow replaces notebook-bound transformations with testable package boundaries. The notebook reads those contracts; it does not reimplement them. The diagram below is a modern repository-owned asset, not an image from the historical archive.

![Supported modern pipeline and evidence lineage](../reports/figures/modern-pipeline-lineage.svg)

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Load package-backed configuration readers and establish repository-relative paths.
#
# COMMENT MAP:
# - This cell imports only repository package modules and standard-library helpers.
# - It does not read generated datasets or model artifacts.
# - It creates the shared ``paths`` and ``root`` variables used by later cells.
# - If this cell fails, the environment was not launched through the project setup
#   expected by Step 0, typically ``uv run --group notebooks jupyter lab``.

from dataclasses import asdict
import json
from pathlib import Path

from ecg_anomaly_detection.benchmark_policy import load_benchmark_policy
from ecg_anomaly_detection.config import RepositoryPaths, load_dataset_config
from ecg_anomaly_detection.evaluation import load_evaluation_config
from ecg_anomaly_detection.labels import load_annotation_mapping
from ecg_anomaly_detection.splitting import load_split_config
from ecg_anomaly_detection.training import load_training_config
from ecg_anomaly_detection.windows import load_window_config

# Discovering paths through the package keeps notebook path handling aligned
# with the CLI and tests instead of reimplementing repository layout logic here.
paths = RepositoryPaths.discover()
root = paths.root

# Return a small audit object instead of printing prose so execution output is
# compact, deterministic, and easy to inspect in saved notebooks.
{
    "repository_root": str(root),
    "package_backed": True,
}


## 2. Historical archive boundary

The notebooks under `archive/original_2022/` are preserved historical artifacts. They are unsupported, may contain stale errors and duplicated exploratory logic, and are not executed by this walkthrough. Their saved metrics remain historical results with known random beat-window split limitations; they do not demonstrate generalization to unseen patients. No historical image is reused here.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Confirm the historical archive boundary without executing archived notebooks.
#
# COMMENT MAP:
# - This cell checks only whether the preserved historical archive directory exists.
# - It does not import, execute, rewrite, or validate unsupported historical notebooks.
# - The output supports the narrative distinction between preserved history and
#   the supported modern package-backed workflow.

archive = root / "archive" / "original_2022"

{
    "archive_exists": archive.is_dir(),
    "archive_role": "preserved and unsupported",
}


## 3. Configuration-driven workflow

Each stage validates a committed TOML contract before touching data. Loading these contracts is deterministic and data-independent.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Load the committed configuration contracts that define the supported workflow.
#
# COMMENT MAP:
# - Each loader validates a versioned TOML file from ``configs/``.
# - The cell is deterministic and data-independent; it does not require local MIT-BIH files.
# - The output reports contract identity rather than model performance.

dataset = load_dataset_config(paths.configs / "mitdb-v1.0.0.toml")
mapping = load_annotation_mapping(paths.configs / "annotation-map-v1.toml")
windowing = load_window_config(paths.configs / "windowing-v1.toml")
splitting = load_split_config(paths.configs / "splitting-v2.toml")
training = load_training_config(paths.configs / "training-baseline-v1.toml")
evaluation = load_evaluation_config(paths.configs / "evaluation-baseline-v1.toml")

{
    "dataset": f"{dataset.slug}@{dataset.version}",
    "records": len(dataset.record_ids),
    "mapping": f"{mapping.name}@{mapping.version}",
    "windowing": f"{windowing.name}@{windowing.version}",
    "splitting": f"{splitting.name}@{splitting.version}",
    "training": f"{training.name}@{training.version}",
    "evaluation": f"{evaluation.name}@{evaluation.version}",
}


## 4. Acquisition and source validation

The acquisition stage retrieves the configured public MIT-BIH release into the ignored raw-data zone, checks the repository-reviewed expected file inventory, and records acquisition and SHA-256 evidence. This notebook deliberately does not download the complete dataset during routine validation. A local full run uses the supported command shown later.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Summarize acquisition contract metadata without downloading the dataset.
#
# COMMENT MAP:
# - This cell reads committed dataset configuration already loaded above.
# - It does not call the acquisition CLI, access the network, or inspect raw ECG records.
# - The ``raw_data_committed`` value documents the repository boundary: raw data stays
#   out of Git and is generated locally through Step 0.

{
    "source": dataset.source_url,
    "sample_rate_hz": dataset.sample_rate_hz,
    "required_files": len(dataset.expected_files),
    "reviewed_source_digests": len(dataset.expected_source_files),
    "raw_data_committed": False,
}


## 5. Annotation mapping

The package applies a closed-world, versioned mapping. Every source symbol must be assigned to a target or explicitly excluded; unknown symbols fail closed. Per-record JSON reports preserve inclusion, exclusion, and target counts.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Show the closed-world annotation mapping contract.
#
# COMMENT MAP:
# - The mapping object comes from the committed TOML contract loaded earlier.
# - Unknown symbols are intentionally fail-closed in the package layer.
# - This cell reports target/exclusion policy only; it does not read annotation files.

{
    "unknown_symbol_policy": mapping.unknown_symbol_policy,
    "targets": {rule.name: list(rule.symbols) for rule in mapping.targets},
    "explicitly_excluded_symbols": list(mapping.excluded_symbols),
}


## 6. Window extraction

The package converts configured time geometry into samples, selects the configured channel, excludes incomplete boundary windows, and retains record ID, annotation location, source symbol, and target lineage. It also reports adjacent overlap rather than hiding it.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Display the configured beat-window extraction geometry and lineage fields.
#
# COMMENT MAP:
# - This reports the committed windowing contract; it does not extract windows.
# - The channel setting is intentionally visible because PIPE-006 concerns channel
#   identity consistency across records.
# - The lineage field list documents what downstream artifacts must preserve.

{
    "pre_seconds": windowing.pre_seconds,
    "post_seconds": windowing.post_seconds,
    "channel_index": windowing.channel_index,
    "boundary_policy": windowing.boundary_policy,
    "lineage_fields": [
        "record_id",
        "center_sample_index",
        "source_symbol",
        "target_value",
    ],
}


## 7. Subject-aware splitting and split diagnostics

Split schema v2 maps records to subjects and assigns complete subjects to deterministic train, validation, and protected test partitions. Package assertions reject subject or record crossover. A generated split-quality summary records disjointness, partition counts, class coverage, ratio deviations, and configured warning/failure thresholds.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Summarize the subject-aware split contract without opening generated shards.
#
# COMMENT MAP:
# - The split contract is committed configuration, not generated data.
# - Subject grouping is the relevant modernization behavior; records should not
#   cross train/validation/test partitions.
# - This output supports evaluation-governance discussion without computing metrics.

{
    "schema_version": splitting.schema_version,
    "strategy": splitting.strategy,
    "seed": splitting.seed,
    "ratios": {
        "train": splitting.train_ratio,
        "validation": splitting.validation_ratio,
        "test": splitting.test_ratio,
    },
    "record_subject_mappings": len(splitting.record_subjects),
}


## 8. Baseline training and validation-only evaluation

Training resolves only training shards from the model-ready index. Evaluation then loads the frozen model and resolves only validation shards, verifying content digests before scoring. Validation metrics are pipeline evidence—not benchmark evidence, protected-test evidence, or proof of generalization.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Report the baseline training/evaluation contract and protected-test boundary.
#
# COMMENT MAP:
# - This cell does not train, score, or load a model.
# - It documents that routine evaluation is validation-only.
# - The protected test partition remains unopened by this walkthrough.

{
    "estimator": training.estimator,
    "training_seed": training.seed,
    "training_partition": "train",
    "evaluation_partition": evaluation.partition,
    "test_partition_accessed": False,
}


## 9. Reproducibility evidence, run manifests, and artifact lineage

A supported full run writes environment, runtime, resource, configuration, report, artifact, and digest evidence beneath an ignored UUID-scoped run directory. The final run manifest links those records. The next cell reads only existing JSON evidence; when local artifacts are absent, it returns an explicit status and continues cleanly.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Inspect optional local run-manifest evidence written by a completed Step 0 run.
#
# COMMENT MAP:
# - This cell reads JSON evidence only if it already exists under ignored artifacts.
# - It does not run the pipeline, clean state, generate metrics, or open protected test data.
# - Missing evidence is an expected state when Step 0 is blocked by PIPE-006 or has
#   not been run yet, so the cell returns status instead of failing.

run_root = paths.artifacts / "runs"

# Sorting by modification time makes "latest" reflect the most recent local run
# rather than lexicographic UUID order.
run_manifests = (
    sorted(
        run_root.glob("*/run-manifest.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if run_root.is_dir()
    else []
)

if run_manifests:
    latest_manifest_path = run_manifests[-1]
    latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))

    evidence_status = {
        "status": "local evidence available",
        "manifest": str(latest_manifest_path.relative_to(root)),
        "schema_version": latest_manifest.get("schema_version"),
        "top_level_fields": sorted(latest_manifest),
    }
else:
    evidence_status = {
        "status": "no local run evidence available",
        "expected_location": "artifacts/runs/<run-id>/run-manifest.json",
        "likely_reason": (
            "Step 0 has not completed artifact generation, or it ended in a controlled "
            "blocked state such as PIPE-006."
        ),
    }

evidence_status


Run the complete supported workflow from the repository root when local data acquisition is intended:

```fish
uv run ecg-data run-pipeline \
  --repository-root . \
  --dataset-config configs/mitdb-v1.0.0.toml \
  --mapping-config configs/annotation-map-v1.toml \
  --window-config configs/windowing-v1.toml \
  --split-config configs/splitting-v2.toml \
  --training-config configs/training-baseline-v1.toml \
  --evaluation-config configs/evaluation-baseline-v1.toml
```

## 10. Benchmark governance boundary

The held-out test partition is intentionally unavailable for routine evaluation. The benchmark policy is fail-closed: test evaluation is disabled and any future access requires separate review, explicit opt-in, immutable lineage, disclosure, and archival controls. This notebook validates that policy without opening data.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Validate the benchmark-governance policy without accessing protected test data.
#
# COMMENT MAP:
# - This cell reads only the committed benchmark policy configuration.
# - Assertions are intentional because a changed policy boundary should stop the walkthrough.
# - The assertions do not open data, compute metrics, or touch generated artifacts.

benchmark_policy = load_benchmark_policy(paths.configs / "benchmark-policy-v1.toml")

# These assertions document the repository governance boundary: routine notebooks
# cannot silently become benchmark/test-evaluation workflows.
assert benchmark_policy.protected_partition == "test"
assert benchmark_policy.test_evaluation_enabled is False
assert benchmark_policy.explicit_future_opt_in_required is True

{
    "policy": f"{benchmark_policy.policy_id}@{benchmark_policy.version}",
    "protected_partition": benchmark_policy.protected_partition,
    "test_evaluation_enabled": benchmark_policy.test_evaluation_enabled,
    "future_opt_in_required": benchmark_policy.explicit_future_opt_in_required,
}


## 11. Portfolio interpretation and limitations

The engineering result is a reproducible, configuration-driven local pipeline with explicit provenance, validation, lineage, subject-aware preparation, testable transformations, and operational evidence. It demonstrates data-engineering and platform-engineering judgment.

It does **not** establish model quality, unseen-population generalization, clinical validity, diagnostic value, medical utility, production readiness, or deployment fitness. Historical notebook metrics remain historical artifacts. Modern validation-only metrics remain development evidence. No protected test metric or supported benchmark is produced here.